# 00 · Setup compartido

Este notebook **no se ejecuta solo**: cada módulo lo llama con `%run ./00_Setup`.

Deja listo para todos los módulos:

| Qué | Para qué |
|---|---|
| Esquema y volumen en Unity Catalog | Un lugar propio y ordenado para el laboratorio |
| Rutas y nombres de tablas | Un solo sitio donde cambiarlos |
| Esquema del JSON de TVMaze | Se define una vez y se reutiliza |
| Funciones auxiliares | `mb()` y `leer_raw()` |

> Así se usa `%run` (slide 18): funciones y variables comunes en un solo notebook.

## 1. Ubicación: catálogo, esquema y volumen

In [ ]:
import time
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]   # 'workspace' en la Free Edition
SCHEMA  = "lab_tvmaze"
VOLUME  = "datos"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA {SCHEMA}")

## 2. Rutas y nombres de tablas

In [ ]:
# Archivos (Volumen)
BASE          = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_PATH      = f"{BASE}/raw/tvmaze"                    # capa RAW: JSON tal cual los entrega la API
RAW_GLOB      = f"{RAW_PATH}/*/*/*/tvmaze.json"         # patrón: año/mes/día/tvmaze.json
FORMATOS_PATH = f"{BASE}/formatos"                      # módulo 4
TMP_PATH      = f"{BASE}/tmp"                           # archivos pequeños de demostración

# Tablas (Delta)
BRONZE   = "bronze_episodios"
SILVER   = "silver_episodios"
GOLD_RED = "gold_resumen_red"
GOLD_DIA = "gold_emisiones_dia"

## 3. Esquema explícito del JSON (slide 23)

Solo los campos que usaremos. Lo que no está aquí, Spark lo ignora.

In [ ]:
ESQUEMA_EPISODIOS = """
    id BIGINT, name STRING, season INT, number INT, type STRING,
    airdate STRING, airtime STRING, airstamp STRING, runtime INT,
    rating STRUCT<average: DOUBLE>,
    _embedded STRUCT<
        show: STRUCT<
            id: BIGINT, name: STRING, type: STRING, language: STRING,
            genres: ARRAY<STRING>, status: STRING, premiered: STRING,
            summary: STRING,
            rating: STRUCT<average: DOUBLE>,
            network: STRUCT<id: BIGINT, name: STRING, country: STRUCT<name: STRING, code: STRING>>,
            webChannel: STRUCT<id: BIGINT, name: STRING, country: STRUCT<name: STRING, code: STRING>>
        >
    >
"""

## 4. Funciones auxiliares

In [ ]:
def fq(tabla):
    """Nombre completo de una tabla: catalogo.esquema.tabla"""
    return f"{CATALOG}.{SCHEMA}.{tabla}"


def bytes_en(ruta):
    """Suma el tamaño de todos los archivos dentro de una carpeta."""
    total = 0
    for f in dbutils.fs.ls(ruta):
        total += bytes_en(f.path) if f.name.endswith("/") else f.size
    return total


def mb(ruta):
    """Tamaño de una carpeta en MB."""
    return round(bytes_en(ruta) / 1024 / 1024, 2)


def leer_raw(ruta=RAW_GLOB):
    """Lee los JSON de RAW con esquema explícito y agrega columnas de trazabilidad."""
    return (spark.read.format("json")
            .option("multiLine", True)                  # cada archivo es un arreglo JSON
            .schema(ESQUEMA_EPISODIOS)
            .load(ruta)
            .select("*",
                    F.col("_metadata.file_path").alias("archivo_origen"),
                    F.current_timestamp().alias("ingestado_en")))

In [ ]:
print("✔ Setup listo")
print(f"  Esquema : {CATALOG}.{SCHEMA}")
print(f"  Archivos: {BASE}")